In [1]:
# ============================================================
# CELL 1: Ternary STE Microbenchmark
# Current vs optimized STE path
# ============================================================
import torch
import torch.nn.functional as F
import time

torch.manual_seed(42)
device = torch.device('cuda')

# Match real model: TernaryLinear weights
# Largest ternary matrix: fused QKV = [dim + 2*kv_dim, dim] = [1024, 768] at 8h/4kv
# MLP fc (relu2 2x) = [1536, 768], MLP proj = [768, 1536]
# Test on MLP fc as it's the biggest
W = torch.randn(1536, 768, device=device, dtype=torch.bfloat16)
GROUP_SIZE = 128
N_WARMUP = 200
N_RUNS = 2000

# --- Current STE (from TernaryLinear.forward) ---
def ste_current(w):
    g = GROUP_SIZE
    w_g = w.reshape(-1, g)
    scale = w_g.abs().mean(-1, keepdim=True).clamp(min=1e-8)
    q = (w_g / scale).round().clamp(-1, 1)
    return w + ((q * scale).reshape(w.shape) - w).detach()

# --- Optimized STE: avoid redundant reshape, fuse ops ---
def ste_optimized(w):
    g = GROUP_SIZE
    w_g = w.reshape(-1, g)
    scale = w_g.abs().mean(-1, keepdim=True).clamp(min=1e-8)
    q = (w_g / scale).round().clamp_(-1, 1)  # in-place clamp
    w_q = (q * scale).view_as(w)
    return w + (w_q - w).detach()

# --- Optimized STE v2: precompute reciprocal, avoid div ---
def ste_v2(w):
    g = GROUP_SIZE
    w_g = w.reshape(-1, g)
    scale = w_g.abs().mean(-1, keepdim=True).clamp(min=1e-8)
    inv_scale = scale.reciprocal()
    q = (w_g * inv_scale).round().clamp(-1, 1)
    return w + ((q * scale).view_as(w) - w).detach()

# Warmup
for _ in range(N_WARMUP):
    _ = ste_current(W)
    _ = ste_optimized(W)
    _ = ste_v2(W)
torch.cuda.synchronize()

for name, fn in [("Current STE", ste_current), ("Optimized STE", ste_optimized), ("STE v2 (reciprocal)", ste_v2)]:
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(N_RUNS):
        _ = fn(W)
    torch.cuda.synchronize()
    elapsed = (time.perf_counter() - t0) / N_RUNS * 1000
    print(f'{name:<25} {elapsed:.4f} ms/call')

# Count: how many STE calls per step at 12L?
# Per block: c_qkv(1) + attn_proj(1) + mlp_fc(1) + mlp_proj(1) = 4
# 12 blocks = 48 STE calls per step
print(f'\n48 STE calls/step (12L): current = {48 * elapsed:.2f} ms overhead estimate')

Current STE               0.0409 ms/call
Optimized STE             0.0424 ms/call
STE v2 (reciprocal)       0.0430 ms/call

48 STE calls/step (12L): current = 2.07 ms overhead estimate


In [2]:
# ============================================================
# CELL 2: .contiguous() necessity check
# Are the tensors actually non-contiguous before flash_attn?
# ============================================================
import torch
import torch.nn.functional as F
import time

device = torch.device('cuda')

B, S, H, KV_H, D = 64, 1024, 8, 4, 96  # batch, seq, heads, kv_heads, head_dim
DIM = H * D  # 768

# Simulate the exact path in CausalSelfAttention.forward:
# 1. c_qkv produces [B, S, q_size + 2*kv_size]
q_size = H * D       # 768
kv_size = KV_H * D   # 384
qkv = torch.randn(B, S, q_size + 2 * kv_size, device=device, dtype=torch.bfloat16)

# 2. Split
q_out, k_out, v_out = qkv.split([q_size, kv_size, kv_size], dim=-1)

# 3. Reshape (NOT transpose — this is your code's path)
q = q_out.reshape(B, S, H, D)
k = k_out.reshape(B, S, KV_H, D)
v = v_out.reshape(B, S, KV_H, D)

print("After reshape (pre-RoPE):")
print(f"  q contiguous: {q.is_contiguous()}, stride: {q.stride()}")
print(f"  k contiguous: {k.is_contiguous()}, stride: {k.stride()}")
print(f"  v contiguous: {v.is_contiguous()}, stride: {v.stride()}")

# 4. Simulate RMSNorm + RoPE (these don't change contiguity of output)
q = F.rms_norm(q, (D,))
k = F.rms_norm(k, (D,))

# RoPE: torch.cat of two halves
half = D // 2
cos = torch.randn(1, S, 1, half, device=device, dtype=torch.bfloat16)
sin = torch.randn(1, S, 1, half, device=device, dtype=torch.bfloat16)
q1, q2 = q[..., :half], q[..., half:]
q = torch.cat((q1 * cos + q2 * sin, q1 * (-sin) + q2 * cos), dim=-1)
k1, k2 = k[..., :half], k[..., half:]
k = torch.cat((k1 * cos + k2 * sin, k1 * (-sin) + k2 * cos), dim=-1)

# 5. Scale
q = q * 2.25  # q_gain broadcast

print("\nAfter RoPE + scale (pre flash_attn):")
print(f"  q contiguous: {q.is_contiguous()}, stride: {q.stride()}")
print(f"  k contiguous: {k.is_contiguous()}, stride: {k.stride()}")
print(f"  v contiguous: {v.is_contiguous()}, stride: {v.stride()}")

# Benchmark the .contiguous() calls
N_WARMUP, N_RUNS = 200, 2000

if q.is_contiguous() and k.is_contiguous() and v.is_contiguous():
    print("\nAll tensors ALREADY contiguous — .contiguous() is a no-op.")
    print("Removing them saves only Python-level overhead (~0.01ms total).")
else:
    for name, tensor in [("q", q), ("k", k), ("v", v)]:
        if not tensor.is_contiguous():
            for _ in range(N_WARMUP):
                _ = tensor.contiguous()
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            for _ in range(N_RUNS):
                _ = tensor.contiguous()
            torch.cuda.synchronize()
            ms = (time.perf_counter() - t0) / N_RUNS * 1000
            print(f"\n  {name}.contiguous(): {ms:.4f} ms/call")
            print(f"  12L cost: {12 * ms:.3f} ms/step")
        else:
            print(f"\n  {name} already contiguous — no cost")

After reshape (pre-RoPE):
  q contiguous: False, stride: (1572864, 1536, 96, 1)
  k contiguous: False, stride: (1572864, 1536, 96, 1)
  v contiguous: False, stride: (1572864, 1536, 96, 1)

After RoPE + scale (pre flash_attn):
  q contiguous: True, stride: (786432, 768, 96, 1)
  k contiguous: True, stride: (393216, 384, 96, 1)
  v contiguous: False, stride: (1572864, 1536, 96, 1)

  q already contiguous — no cost

  k already contiguous — no cost

  v.contiguous(): 0.0650 ms/call
  12L cost: 0.780 ms/step


In [3]:
# ============================================================
# CELL 3: RoPE implementation comparison
# Current (half-split + cat) vs interleaved vs complex multiply
# ============================================================
import torch
import time

device = torch.device('cuda')
B, S, H, D = 64, 1024, 8, 96
half = D // 2
N_WARMUP, N_RUNS = 200, 2000

q = torch.randn(B, S, H, D, device=device, dtype=torch.bfloat16)
cos = torch.randn(1, S, 1, half, device=device, dtype=torch.bfloat16)
sin = torch.randn(1, S, 1, half, device=device, dtype=torch.bfloat16)

# --- Current: half-split + cat ---
def rope_cat(x, cos, sin):
    x1, x2 = x[..., :half], x[..., half:]
    return torch.cat((x1 * cos + x2 * sin, x1 * (-sin) + x2 * cos), dim=-1)

# --- Alternative: stack + reshape (avoids cat, single kernel) ---
def rope_stack(x, cos, sin):
    x1, x2 = x[..., :half], x[..., half:]
    r1 = x1 * cos + x2 * sin
    r2 = x1 * (-sin) + x2 * cos
    # Interleave instead of cat — different layout but same info
    return torch.stack((r1, r2), dim=-1).reshape_as(x)

# --- Alternative: in-place with pre-allocated output ---
out_buf = torch.empty_like(q)
def rope_inplace(x, cos, sin):
    x1, x2 = x[..., :half], x[..., half:]
    out_buf[..., :half] = x1 * cos + x2 * sin
    out_buf[..., half:] = x1 * (-sin) + x2 * cos
    return out_buf

# --- Alternative: complex multiply (pairs as complex numbers) ---
cos_full = torch.randn(1, S, 1, half, device=device, dtype=torch.float32)
sin_full = torch.randn(1, S, 1, half, device=device, dtype=torch.float32)
def rope_complex(x, cos, sin):
    x_f = x.float()
    x1, x2 = x_f[..., :half], x_f[..., half:]
    # Treat (x1, x2) as complex: (x1 + ix2) * (cos + isin)
    r1 = x1 * cos - x2 * sin  # Note: standard rotation, slightly different from your convention
    r2 = x1 * sin + x2 * cos
    return torch.cat((r1, r2), dim=-1).to(x.dtype)

for name, fn, c, s in [
    ("Current (cat)", rope_cat, cos, sin),
    ("Stack+reshape", rope_stack, cos, sin),
    ("In-place buf", rope_inplace, cos, sin),
    ("Complex (fp32)", rope_complex, cos_full, sin_full),
]:
    for _ in range(N_WARMUP):
        _ = fn(q, c, s)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(N_RUNS):
        _ = fn(q, c, s)
    torch.cuda.synchronize()
    ms = (time.perf_counter() - t0) / N_RUNS * 1000
    # 2 RoPE calls per attn (Q and K) * 12 layers
    print(f'{name:<20} {ms:.4f} ms/call   24 calls/step: {24*ms:.3f} ms')

Current (cat)        0.5160 ms/call   24 calls/step: 12.385 ms
Stack+reshape        0.6122 ms/call   24 calls/step: 14.693 ms
In-place buf         0.5630 ms/call   24 calls/step: 13.512 ms
Complex (fp32)       0.9905 ms/call   24 calls/step: 23.772 ms


In [4]:
# ============================================================
# CELL 4: Softcap polynomial vs alternatives
# ============================================================
import torch
import time

device = torch.device('cuda')
N_WARMUP, N_RUNS = 200, 2000

# Logits shape: [batch*seq, vocab] = [64*1024, 8192] = [65536, 8192]
logits = torch.randn(65536, 8192, device=device, dtype=torch.bfloat16)
S = 10.0  # LOGIT_SOFTCAP

# --- Current: 5th order polynomial ---
def softcap_poly5(logits):
    x_sc = torch.clamp(logits / S, -2.0, 2.0)
    x2 = x_sc * x_sc
    return S * torch.clamp(x_sc * (1.0 - x2 / 3.0 + x2 * x2 / 15.0), -1.0, 1.0)

# --- Simpler: 3rd order (x - x^3/3, clamped) ---
def softcap_poly3(logits):
    x_sc = torch.clamp(logits / S, -2.0, 2.0)
    return S * torch.clamp(x_sc * (1.0 - x_sc * x_sc / 3.0), -1.0, 1.0)

# --- Simplest: hardtanh (clamp only, no polynomial) ---
def softcap_hardtanh(logits):
    return torch.clamp(logits, -S, S)

# --- Tanh (for reference — known to be slow due to exp) ---
def softcap_tanh(logits):
    return S * torch.tanh(logits / S)

# Check approximation quality first
test = torch.linspace(-25, 25, 1000, device=device)
ref = S * torch.tanh(test / S)
for name, fn in [("Poly5", softcap_poly5), ("Poly3", softcap_poly3), ("Hardtanh", softcap_hardtanh)]:
    approx = fn(test)
    max_err = (approx - ref).abs().max().item()
    print(f'{name:<12} max_err vs tanh: {max_err:.4f}')

print()

for name, fn in [
    ("Current poly5", softcap_poly5),
    ("Poly3", softcap_poly3),
    ("Hardtanh", softcap_hardtanh),
    ("Tanh", softcap_tanh),
]:
    for _ in range(N_WARMUP):
        _ = fn(logits)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(N_RUNS):
        _ = fn(logits)
    torch.cuda.synchronize()
    ms = (time.perf_counter() - t0) / N_RUNS * 1000
    # 1 softcap call per step
    print(f'{name:<18} {ms:.4f} ms/call')

Poly5        max_err vs tanh: 0.6538
Poly3        max_err vs tanh: 16.5328
Hardtanh     max_err vs tanh: 2.3754

Current poly5      8.4287 ms/call
Poly3              5.9798 ms/call
Hardtanh           0.7060 ms/call
Tanh               2.1184 ms/call


In [5]:
# ============================================================
# CELL 5: Cross-entropy + Z-loss fusion
# Current: separate logsumexp + cross_entropy (both compute LSE)
# Test: extract LSE from manual cross-entropy to avoid double compute
# ============================================================
import torch
import torch.nn.functional as F
import time

device = torch.device('cuda')
N_WARMUP, N_RUNS = 200, 2000

# [batch*seq, vocab]
logits = torch.randn(65536, 8192, device=device, dtype=torch.float32)
targets = torch.randint(0, 8192, (65536,), device=device)

# --- Current: separate logsumexp + cross_entropy ---
def loss_current(logits, targets):
    lse = torch.logsumexp(logits, dim=-1)
    z_loss = 1e-4 * (lse ** 2).mean()
    ce = F.cross_entropy(logits, targets, reduction="mean")
    return ce + z_loss

# --- Fused: manual CE that reuses the LSE ---
def loss_fused(logits, targets):
    lse = torch.logsumexp(logits, dim=-1)
    # CE = -logits[target] + logsumexp = -(logits[target] - lse)
    target_logits = logits.gather(1, targets.unsqueeze(1)).squeeze(1)
    ce = (lse - target_logits).mean()
    z_loss = 1e-4 * (lse ** 2).mean()
    return ce + z_loss

# --- Verify correctness ---
l1 = loss_current(logits, targets)
l2 = loss_fused(logits, targets)
print(f"Current: {l1.item():.6f}")
print(f"Fused:   {l2.item():.6f}")
print(f"Diff:    {abs(l1.item() - l2.item()):.8f}")
print()

for name, fn in [("Current (separate)", loss_current), ("Fused (shared LSE)", loss_fused)]:
    for _ in range(N_WARMUP):
        _ = fn(logits, targets)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(N_RUNS):
        _ = fn(logits, targets)
    torch.cuda.synchronize()
    ms = (time.perf_counter() - t0) / N_RUNS * 1000
    print(f'{name:<25} {ms:.4f} ms/call')

print("\nNote: savings here are per-step (1 call). Also check backward pass cost.")

# --- Backward pass timing ---
print("\nWith backward pass:")
for name, fn in [("Current (separate)", loss_current), ("Fused (shared LSE)", loss_fused)]:
    logits_p = logits.clone().requires_grad_(True)
    for _ in range(N_WARMUP):
        l = fn(logits_p, targets)
        l.backward()
        logits_p.grad = None
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(N_RUNS):
        l = fn(logits_p, targets)
        l.backward()
        logits_p.grad = None
    torch.cuda.synchronize()
    ms = (time.perf_counter() - t0) / N_RUNS * 1000
    print(f'{name:<25} {ms:.4f} ms/call (fwd+bwd)')

Current: 9.519403
Fused:   9.519403
Diff:    0.00000000

Current (separate)        6.7477 ms/call
Fused (shared LSE)        4.6260 ms/call

Note: savings here are per-step (1 call). Also check backward pass cost.

With backward pass:
Current (separate)        16.5612 ms/call (fwd+bwd)
Fused (shared LSE)        12.3295 ms/call (fwd+bwd)
